# WINNIIO — Tokyo MRO Digital Twin: Sionna RF Propagation

Physics-based 3D ray-tracing RF propagation for Altiostar/Rakuten Symphony MRO use case.

**What this does:**
1. Pulls Tokyo 3D buildings from OpenStreetMap (auto-tiled)
2. Places your 22 cell towers at exact coordinates from the Cesium demo
3. Runs GPU ray-tracing propagation (Sionna RT)
4. Generates SINR/RSS coverage heatmaps
5. Exports GeoJSON for Cesium overlay

**Runtime:** Google Colab (free T4 GPU) — ~15 min total

---
*WINNIIO Spatial Intelligence · Apache 2.0 · April 2026*

## 1. Install Sionna + Dependencies

In [ ]:
# Run this cell first — takes ~3-5 min on Colab
!pip install sionna
!pip install mitsuba
!pip install geojson
!pip install osmnx  # OpenStreetMap building pull
!pip install trimesh  # mesh processing
!pip install shapely
print('\n✅ All dependencies installed.')

## 2. Verify GPU

In [ ]:
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'✅ GPU: {gpu} ({mem:.1f} GB)')
else:
    print('⚠️ No GPU — go to Runtime > Change runtime type > T4 GPU')

## 3. Define Tokyo Cell Tower Network

22 towers from the WINNIIO Cesium demo — exact coordinates, bands, power, azimuths.

In [ ]:
import numpy as np

# All 22 towers from altiostar-tokyo-demo.html
towers = [
    # Shinjuku cluster
    {'id': 'SJK-001', 'name': 'Shinjuku Station West',     'lat': 35.6896, 'lng': 139.6982, 'height': 45,  'band': 'n77',  'freq_ghz': 3.7,  'power_dbm': 40, 'sectors': 3, 'azimuth': [0, 120, 240], 'tilt': 4,  'status': 'active'},
    {'id': 'SJK-002', 'name': 'Nishi-Shinjuku Tower',      'lat': 35.6935, 'lng': 139.6917, 'height': 120, 'band': 'n77',  'freq_ghz': 3.7,  'power_dbm': 43, 'sectors': 3, 'azimuth': [30, 150, 270], 'tilt': 6, 'status': 'active'},
    {'id': 'SJK-003', 'name': 'Shinjuku Gyoen South',      'lat': 35.6852, 'lng': 139.7100, 'height': 35,  'band': 'n78',  'freq_ghz': 3.5,  'power_dbm': 37, 'sectors': 3, 'azimuth': [10, 130, 250], 'tilt': 3, 'status': 'active'},
    {'id': 'SJK-004', 'name': 'Kabukicho North',           'lat': 35.6955, 'lng': 139.7030, 'height': 28,  'band': 'n257', 'freq_ghz': 28.0, 'power_dbm': 30, 'sectors': 4, 'azimuth': [0, 90, 180, 270], 'tilt': 8, 'status': 'active'},
    # Shibuya cluster
    {'id': 'SBY-001', 'name': 'Shibuya Crossing',          'lat': 35.6595, 'lng': 139.7004, 'height': 55,  'band': 'n77',  'freq_ghz': 3.7,  'power_dbm': 43, 'sectors': 3, 'azimuth': [20, 140, 260], 'tilt': 5, 'status': 'active'},
    {'id': 'SBY-002', 'name': 'Shibuya Stream',            'lat': 35.6565, 'lng': 139.7030, 'height': 80,  'band': 'n78',  'freq_ghz': 3.5,  'power_dbm': 40, 'sectors': 3, 'azimuth': [0, 120, 240], 'tilt': 4, 'status': 'active'},
    {'id': 'SBY-003', 'name': 'Harajuku Meiji',            'lat': 35.6702, 'lng': 139.7027, 'height': 25,  'band': 'n257', 'freq_ghz': 28.0, 'power_dbm': 28, 'sectors': 4, 'azimuth': [0, 90, 180, 270], 'tilt': 10, 'status': 'active'},
    # Minato cluster
    {'id': 'MNT-001', 'name': 'Tokyo Tower Site',          'lat': 35.6586, 'lng': 139.7454, 'height': 333, 'band': 'n77',  'freq_ghz': 3.7,  'power_dbm': 46, 'sectors': 6, 'azimuth': [0, 60, 120, 180, 240, 300], 'tilt': 8, 'status': 'active'},
    {'id': 'MNT-002', 'name': 'Roppongi Hills',            'lat': 35.6605, 'lng': 139.7292, 'height': 54,  'band': 'n77',  'freq_ghz': 3.7,  'power_dbm': 40, 'sectors': 3, 'azimuth': [15, 135, 255], 'tilt': 4, 'status': 'active'},
    {'id': 'MNT-003', 'name': 'Toranomon Hills',           'lat': 35.6670, 'lng': 139.7495, 'height': 52,  'band': 'n78',  'freq_ghz': 3.5,  'power_dbm': 40, 'sectors': 3, 'azimuth': [0, 120, 240], 'tilt': 5, 'status': 'active'},
    {'id': 'MNT-004', 'name': 'Azabudai Hills',            'lat': 35.6594, 'lng': 139.7370, 'height': 64,  'band': 'n257', 'freq_ghz': 28.0, 'power_dbm': 33, 'sectors': 4, 'azimuth': [45, 135, 225, 315], 'tilt': 7, 'status': 'active'},
    # Central cluster
    {'id': 'CTR-001', 'name': 'Tokyo Station Marunouchi',  'lat': 35.6812, 'lng': 139.7671, 'height': 50,  'band': 'n77',  'freq_ghz': 3.7,  'power_dbm': 43, 'sectors': 3, 'azimuth': [0, 120, 240], 'tilt': 4, 'status': 'active'},
    {'id': 'CTR-002', 'name': 'Ginza Chuo-dori',           'lat': 35.6717, 'lng': 139.7649, 'height': 38,  'band': 'n78',  'freq_ghz': 3.5,  'power_dbm': 37, 'sectors': 3, 'azimuth': [30, 150, 270], 'tilt': 3, 'status': 'active'},
    {'id': 'CTR-003', 'name': 'Akihabara Electric',        'lat': 35.6984, 'lng': 139.7731, 'height': 30,  'band': 'n77',  'freq_ghz': 3.7,  'power_dbm': 40, 'sectors': 3, 'azimuth': [0, 120, 240], 'tilt': 4, 'status': 'active'},
    # Ikebukuro cluster
    {'id': 'IKB-001', 'name': 'Ikebukuro Sunshine',        'lat': 35.7295, 'lng': 139.7185, 'height': 60,  'band': 'n77',  'freq_ghz': 3.7,  'power_dbm': 43, 'sectors': 3, 'azimuth': [10, 130, 250], 'tilt': 5, 'status': 'active'},
    {'id': 'IKB-002', 'name': 'Ikebukuro West Gate',       'lat': 35.7310, 'lng': 139.7109, 'height': 35,  'band': 'n78',  'freq_ghz': 3.5,  'power_dbm': 37, 'sectors': 3, 'azimuth': [0, 120, 240], 'tilt': 3, 'status': 'active'},
    # Others
    {'id': 'SGW-001', 'name': 'Shinagawa Station',         'lat': 35.6284, 'lng': 139.7387, 'height': 45,  'band': 'n77',  'freq_ghz': 3.7,  'power_dbm': 43, 'sectors': 3, 'azimuth': [0, 120, 240], 'tilt': 4, 'status': 'active'},
    {'id': 'ODB-001', 'name': 'Odaiba Telecom Center',     'lat': 35.6190, 'lng': 139.7760, 'height': 70,  'band': 'n77',  'freq_ghz': 3.7,  'power_dbm': 46, 'sectors': 6, 'azimuth': [0, 60, 120, 180, 240, 300], 'tilt': 6, 'status': 'active'},
    # Handover failure zones (degraded sites)
    {'id': 'HO-001',  'name': 'Yamanote Ueno Junction',    'lat': 35.7135, 'lng': 139.7770, 'height': 22,  'band': 'n77',  'freq_ghz': 3.7,  'power_dbm': 34, 'sectors': 2, 'azimuth': [90, 270], 'tilt': 2, 'status': 'degraded'},
    {'id': 'HO-002',  'name': 'Meguro River Corridor',     'lat': 35.6420, 'lng': 139.7150, 'height': 18,  'band': 'n78',  'freq_ghz': 3.5,  'power_dbm': 30, 'sectors': 2, 'azimuth': [0, 180], 'tilt': 2, 'status': 'degraded'},
    {'id': 'HO-003',  'name': 'Shuto Expressway C1 Loop',  'lat': 35.6750, 'lng': 139.7550, 'height': 15,  'band': 'n77',  'freq_ghz': 3.7,  'power_dbm': 33, 'sectors': 2, 'azimuth': [45, 225], 'tilt': 2, 'status': 'degraded'},
]

print(f'✅ {len(towers)} towers loaded')
print(f'   Active: {sum(1 for t in towers if t["status"]=="active")}')
print(f'   Degraded: {sum(1 for t in towers if t["status"]=="degraded")}')
print(f'   Bands: n77 (3.7GHz), n78 (3.5GHz), n257 (28GHz mmWave)')

## 4. Pull Tokyo 3D Buildings from OpenStreetMap

Bounding box covers Shinjuku → Tokyo Station → Odaiba.

In [ ]:
import osmnx as ox
import trimesh
import os

# Tokyo bounding box covering all 22 tower sites
# South-West to North-East
SOUTH, WEST = 35.615, 139.685
NORTH, EAST = 35.735, 139.785

print(f'Pulling buildings for Tokyo area ({SOUTH},{WEST}) to ({NORTH},{EAST})...')
print('This takes 2-4 min depending on OSM server load...')

# Pull building footprints with heights
tags = {'building': True}
gdf = ox.features_from_bbox(bbox=(NORTH, SOUTH, EAST, WEST), tags=tags)

# Filter to polygons only
gdf = gdf[gdf.geometry.type.isin(['Polygon', 'MultiPolygon'])].copy()

# Extract heights (OSM uses 'height' or 'building:levels')
def get_height(row):
    if 'height' in row and row['height'] is not None:
        try:
            return float(str(row['height']).replace('m', '').strip())
        except:
            pass
    if 'building:levels' in row and row['building:levels'] is not None:
        try:
            return float(row['building:levels']) * 3.0  # 3m per floor
        except:
            pass
    return 10.0  # default 10m for unknown

gdf['bldg_height'] = gdf.apply(get_height, axis=1)

print(f'✅ {len(gdf)} buildings pulled')
print(f'   Height range: {gdf["bldg_height"].min():.0f}m — {gdf["bldg_height"].max():.0f}m')
print(f'   Mean height: {gdf["bldg_height"].mean():.1f}m')

## 5. Convert Buildings to Mitsuba 3 Scene (Sionna Format)

In [ ]:
from shapely.geometry import Polygon as ShapelyPolygon
import json

os.makedirs('tokyo_scene', exist_ok=True)

# Reference point (center of our area) for local coordinates
REF_LAT = (SOUTH + NORTH) / 2
REF_LNG = (WEST + EAST) / 2

# Lat/lng to local meters (simple Mercator approximation, fine for city scale)
def to_local(lat, lng):
    x = (lng - REF_LNG) * 111320 * np.cos(np.radians(REF_LAT))
    y = (lat - REF_LAT) * 110540
    return x, y

def polygon_to_obj(polygon, height, idx):
    """Extrude a 2D polygon to 3D OBJ geometry."""
    coords = list(polygon.exterior.coords)
    if len(coords) < 4:
        return None
    coords = coords[:-1]  # remove closing duplicate
    
    vertices = []
    faces = []
    n = len(coords)
    
    # Bottom and top vertices
    for c in coords:
        x, y = to_local(c[1], c[0])  # shapely is (lng, lat)
        vertices.append(f'v {x:.2f} 0.0 {y:.2f}')      # bottom
        vertices.append(f'v {x:.2f} {height:.2f} {y:.2f}')  # top
    
    # Side faces
    for i in range(n):
        j = (i + 1) % n
        b1, t1 = i*2+1, i*2+2
        b2, t2 = j*2+1, j*2+2
        faces.append(f'f {b1} {b2} {t2} {t1}')
    
    # Top face (simple fan triangulation)
    top_indices = [i*2+2 for i in range(n)]
    for i in range(1, n-1):
        faces.append(f'f {top_indices[0]} {top_indices[i]} {top_indices[i+1]}')
    
    return '\n'.join(vertices + faces)

# Convert buildings to OBJ (batch — max 5000 for Colab memory)
MAX_BUILDINGS = 5000
sample = gdf.nlargest(MAX_BUILDINGS, 'bldg_height')  # prioritize tallest

obj_parts = []
offset = 0
for idx, row in enumerate(sample.itertuples()):
    geom = row.geometry
    if geom.geom_type == 'MultiPolygon':
        geom = list(geom.geoms)[0]  # take largest
    obj = polygon_to_obj(geom, row.bldg_height, idx)
    if obj:
        obj_parts.append(f'o building_{idx}\n{obj}')

obj_content = '\n\n'.join(obj_parts)
with open('tokyo_scene/buildings.obj', 'w') as f:
    f.write(obj_content)

print(f'✅ {len(obj_parts)} buildings exported to OBJ')
print(f'   File: tokyo_scene/buildings.obj ({os.path.getsize("tokyo_scene/buildings.obj")/1e6:.1f} MB)')

In [ ]:
# Generate Mitsuba 3 XML scene file for Sionna
mitsuba_xml = '''<?xml version="1.0" encoding="utf-8"?>
<scene version="3.0.0">
    <integrator type="path"/>

    <!-- ITU radio materials for RF propagation -->
    <bsdf type="diffuse" id="mat_concrete">
        <rgb name="reflectance" value="0.5, 0.5, 0.5"/>
    </bsdf>

    <!-- Ground plane -->
    <shape type="rectangle">
        <transform name="to_world">
            <scale x="10000" y="10000" z="1"/>
        </transform>
        <ref id="mat_concrete"/>
    </shape>

    <!-- Tokyo buildings from OSM -->
    <shape type="obj">
        <string name="filename" value="buildings.obj"/>
        <ref id="mat_concrete"/>
    </shape>
</scene>
'''

with open('tokyo_scene/tokyo.xml', 'w') as f:
    f.write(mitsuba_xml)

print('✅ Mitsuba scene: tokyo_scene/tokyo.xml')

## 6. Run Sionna Ray-Tracing

Place transmitters at tower positions, compute coverage heatmap.

In [ ]:
import sionna
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, RadioMapSolver

# Load the Tokyo scene
scene = load_scene('tokyo_scene/tokyo.xml')

# Configure antenna arrays (typical macro cell)
scene.tx_array = PlanarArray(
    num_rows=4, num_cols=4,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='tr38901', polarization='cross'
)
scene.rx_array = PlanarArray(
    num_rows=1, num_cols=1,
    vertical_spacing=0.5, horizontal_spacing=0.5,
    pattern='dipole', polarization='V'
)

# Place transmitters at tower locations (sub-6 GHz only for this demo)
sub6_towers = [t for t in towers if t['freq_ghz'] < 10]

for t in sub6_towers:
    x, y = to_local(t['lat'], t['lng'])
    scene.add(Transmitter(
        name=t['id'],
        position=[x, y, t['height']],
        power_dbm=t['power_dbm']
    ))

print(f'✅ {len(sub6_towers)} sub-6GHz transmitters placed')
print(f'   Skipped {len(towers) - len(sub6_towers)} mmWave towers (n257 — separate sim needed)')

# Set carrier frequency
scene.frequency = 3.7e9  # n77 dominant band
print(f'   Carrier: {scene.frequency/1e9:.1f} GHz')

In [ ]:
# Compute coverage radio map
print('Computing coverage map (this takes 3-8 min on T4)...')

rm_solver = RadioMapSolver()

# Coverage map parameters
radio_map = rm_solver(
    scene=scene,
    max_depth=5,           # max reflections/diffractions
    cell_size=[5.0, 5.0],  # 5m x 5m grid resolution
    samples_per_tx=10**5,  # rays per transmitter (100K — balance speed vs accuracy)
    rx_height=1.5,         # UE height (pedestrian)
)

print('✅ Radio map computed!')
print(f'   Grid: {radio_map.shape}')

## 7. Visualize Coverage

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Extract path gain / RSS data
# radio_map contains path gain in dB
path_gain_db = radio_map.numpy()

# Convert to RSS (received signal strength)
# RSS = Tx Power (dBm) + Path Gain (dB)
# Use max Tx power as reference
rss = path_gain_db + 43  # 43 dBm typical macro

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Custom colormap matching the Cesium demo colors
cmap = mcolors.LinearSegmentedColormap.from_list('mro', [
    '#ff5252',  # weak (red)
    '#ffca28',  # moderate (yellow)
    '#00e676',  # strong (green)
])

# Plot 1: RSS heatmap
ax = axes[0]
im = ax.imshow(rss.squeeze(), cmap=cmap, vmin=-120, vmax=-60,
               origin='lower', aspect='equal')
ax.set_title('RSS Coverage (dBm) — 3.7 GHz', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, label='RSS (dBm)', shrink=0.8)

# Mark tower positions
for t in sub6_towers:
    x, y = to_local(t['lat'], t['lng'])
    # Convert to grid coordinates (approximate)
    gx = (x - (-5500)) / 5.0  # adjust based on grid offset
    gy = (y - (-6600)) / 5.0
    color = 'red' if t['status'] == 'degraded' else 'white'
    ax.plot(gx, gy, 'v', color=color, markersize=8, markeredgecolor='black', markeredgewidth=0.5)
    ax.annotate(t['id'], (gx, gy), fontsize=6, color='white',
                xytext=(3, 3), textcoords='offset points')

# Plot 2: SINR (best server)
ax = axes[1]
# For multi-TX SINR, we'd need per-TX maps — this is simplified
ax.imshow(rss.squeeze(), cmap='RdYlGn', vmin=-120, vmax=-60,
          origin='lower', aspect='equal')
ax.set_title('Coverage Quality — HO Problem Areas', fontsize=14, fontweight='bold')

# Mark HO failure zones
for t in towers:
    if t['status'] == 'degraded':
        x, y = to_local(t['lat'], t['lng'])
        gx = (x - (-5500)) / 5.0
        gy = (y - (-6600)) / 5.0
        circle = plt.Circle((gx, gy), 50, fill=False, color='red', linewidth=2, linestyle='--')
        ax.add_patch(circle)
        ax.annotate(f'{t["id"]}\nHO FAILURE', (gx, gy), fontsize=7, color='red',
                    ha='center', fontweight='bold')

plt.suptitle('WINNIIO — Tokyo MRO Digital Twin: Sionna Ray-Tracing Coverage',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('tokyo_coverage_map.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('✅ Saved: tokyo_coverage_map.png')

## 8. Export GeoJSON for Cesium Overlay

Converts the coverage grid back to lat/lng coordinates for overlaying on the 3D demo.

In [ ]:
import geojson

def to_latlng(x, y):
    """Convert local meters back to lat/lng."""
    lng = x / (111320 * np.cos(np.radians(REF_LAT))) + REF_LNG
    lat = y / 110540 + REF_LAT
    return lat, lng

# Downsample for reasonable GeoJSON size (every 4th cell)
step = 4
features = []

rss_grid = rss.squeeze()
rows, cols = rss_grid.shape

for r in range(0, rows, step):
    for c in range(0, cols, step):
        val = float(rss_grid[r, c])
        if val < -130:  # skip very weak
            continue
        
        # Grid position to local meters (adjust offsets based on actual grid)
        x = c * 5.0 + (-5500)  # adjust based on grid
        y = r * 5.0 + (-6600)
        lat, lng = to_latlng(x, y)
        
        # Color based on signal strength
        if val > -80:
            color = '#00e676'  # strong
            quality = 'strong'
        elif val > -95:
            color = '#ffca28'  # moderate
            quality = 'moderate'
        else:
            color = '#ff5252'  # weak
            quality = 'weak'
        
        # Create a small square polygon for each grid cell
        half = step * 5.0 / 2  # half cell size in meters
        lat1, lng1 = to_latlng(x - half, y - half)
        lat2, lng2 = to_latlng(x + half, y + half)
        
        poly = geojson.Polygon([[
            [lng1, lat1], [lng2, lat1], [lng2, lat2], [lng1, lat2], [lng1, lat1]
        ]])
        
        feature = geojson.Feature(
            geometry=poly,
            properties={
                'rss_dbm': round(val, 1),
                'quality': quality,
                'fill': color,
                'fill-opacity': 0.4,
                'stroke': color,
                'stroke-width': 0,
            }
        )
        features.append(feature)

fc = geojson.FeatureCollection(features)

with open('tokyo_rf_coverage.geojson', 'w') as f:
    geojson.dump(fc, f)

print(f'✅ GeoJSON exported: tokyo_rf_coverage.geojson')
print(f'   {len(features)} coverage cells')
print(f'   Download this file and load in the Cesium demo!')

## 9. 3D Scene Preview

In [ ]:
# Render the 3D scene with ray paths
scene.preview()

## 10. Per-Tower Coverage Analysis

Shows coverage radius and handover boundary for each tower — useful for MRO parameter tuning.

In [ ]:
# Summary table
print('Tower Coverage Summary')
print('=' * 80)
print(f'{"ID":<10} {"Name":<28} {"Band":<8} {"Power":<8} {"Height":<8} {"Status":<10}')
print('-' * 80)
for t in towers:
    status_icon = '✅' if t['status'] == 'active' else '⚠️'
    print(f'{t["id"]:<10} {t["name"]:<28} {t["band"]:<8} {t["power_dbm"]}dBm{"":<3} {t["height"]}m{"":<5} {status_icon} {t["status"]}')

print('\n' + '=' * 80)
print('\nNext steps for Phase 1 Workshop:')
print('  1. Replace OSM buildings with PLATEAU CityGML LOD2 (higher fidelity)')
print('  2. Calibrate propagation against Altiostar drive test data')
print('  3. Run per-sector coverage with actual antenna patterns')
print('  4. Identify real HO failure zones from E2 measurement reports')
print('  5. Train RL agent on calibrated digital twin')

---

### How to use the GeoJSON in the Cesium demo

Download `tokyo_rf_coverage.geojson` and add this to your `altiostar-tokyo-demo.html`:

```javascript
// Load Sionna coverage overlay
Cesium.GeoJsonDataSource.load('tokyo_rf_coverage.geojson', {
  stroke: Cesium.Color.TRANSPARENT,
  fill: Cesium.Color.WHITE.withAlpha(0.3),
  clampToGround: true
}).then(function(ds) {
  viewer.dataSources.add(ds);
  // Color each cell by its RSS value
  ds.entities.values.forEach(function(e) {
    var color = Cesium.Color.fromCssColorString(e.properties.fill.getValue());
    e.polygon.material = color.withAlpha(0.35);
  });
});
```

---

*WINNIIO Spatial Intelligence — "Experts to data, not data to experts"*

*Sionna: Apache 2.0 | PLATEAU: Open Government Data | CesiumJS: Apache 2.0*

*No vendor lock-in. No proprietary ASICs. Just physics.*